# Lab 37 (solution): Evaluation gates for RAG

Reference implementation. Two pieces of evaluation infrastructure: an **LLM-as-judge** scorer that drops into [Lab 34](../34-rag-pattern-head-to-head/)'s harness in place of token-presence scoring, and an **eval gate** that wraps the [Lab 36](../36-training-the-router/) router so routing regressions fail CI.

Pattern source: Zheng et al. 2023, *Judging LLM-as-a-judge* ([arXiv:2306.05685](https://arxiv.org/abs/2306.05685)); the gate makes the [evaluation framework](../../concepts/evaluation/rag-evaluation-framework.md)'s `eval_gate` concrete.

## Step 0: Setup

In [ ]:
import json
import os
import pathlib
import re
from dotenv import load_dotenv
here=pathlib.Path.cwd()
for parent in [here,*here.parents]:
    if (parent/".env.example").exists():
        load_dotenv(parent/".env")
        break
assert os.getenv("OPENAI_API_KEY") or os.getenv("ANTHROPIC_API_KEY")
PROVIDER="openai"
# Use a STRONGER model as judge than as generator where you can afford it - a judge
# should not share the generator's blind spots.
JUDGE_MODEL={"openai":"gpt-4o","anthropic":"claude-opus-4-8"}[PROVIDER]
GEN_MODEL  ={"openai":"gpt-4o-mini","anthropic":"claude-haiku-4-5-20251001"}[PROVIDER]
print(f"judge={JUDGE_MODEL}  generator={GEN_MODEL}")

In [ ]:
def chat(messages, model, temperature=0.0):
    if PROVIDER=="openai":
        from openai import OpenAI
        r=OpenAI().chat.completions.create(model=model,messages=messages,temperature=temperature)
        return r.choices[0].message.content or ""
    from anthropic import Anthropic
    system=next((m["content"] for m in messages if m["role"]=="system"),"")
    ns=[m for m in messages if m["role"]!="system"]
    r=Anthropic().messages.create(model=model,system=system,messages=ns,max_tokens=512,temperature=temperature)
    return "".join(b.text for b in r.content if hasattr(b,"text"))

## Step 1: The LLM judge

A pointwise scorer with an explicit rubric and structured JSON output, parsed defensively.

In [ ]:
def parse_judge(raw:
    str) -> dict:
    """Robustly parse the judge's JSON. Never trust a model to emit clean JSON."""
    raw=re.sub(r"^```(json)?|```$","",raw.strip(),flags=re.MULTILINE).strip()
    try:
        obj=json.loads(raw)
    except Exception:
        m=re.search(r"\{.*\}",raw,re.DOTALL)
        obj=json.loads(m.group(0)) if m else {}
    return {"correct": bool(obj.get("correct",False)),
            "faithful": bool(obj.get("faithful",False)),
            "score": int(obj.get("score",0)),
            "reason": str(obj.get("reason",""))[:200]}

JUDGE_RUBRIC=("You are a strict evaluator. Given a question, a reference answer, and a "
  "candidate answer, judge the candidate. Return JSON only:\n"
  '{"correct": true/false, "faithful": true/false, "score": 1-5, "reason": "..."}\n'
  "- correct: does the candidate convey the reference answer's facts (paraphrase is fine)?\n"
  "- faithful: does it avoid claims not supported by the reference (no fabrication)?\n"
  "- score: 1 (wrong/fabricated) to 5 (correct and faithful).\n"
  "If the reference says the question is unanswerable, a candidate that abstains is correct.")

def llm_judge(query, candidate, reference):
    raw=chat([{"role":"system","content":JUDGE_RUBRIC},
              {"role":"user","content":f"Question: {query}\nReference: {reference}\nCandidate: {candidate}"}],
             model=JUDGE_MODEL)
    return parse_judge(raw)

## Step 2: Validate the judge on adversarial fixtures

The cases where token-presence scoring is known to fail: paraphrase, verbose fabrication, abstention.

In [ ]:
# Validate the judge on adversarial fixtures BEFORE trusting it - cases where simple
# token-presence scoring is known to fail. (query, candidate, reference, token_expected)
fixtures=[
 ("Who leads the Helix Lab?", "Dr. Rao heads the Helix Lab.", "Aanya Rao leads the Helix Lab.", ["Aanya Rao"]),
 ("Who leads the Helix Lab?", "The Helix Lab is led by Aanya Rao, who also won three Nobel Prizes.", "Aanya Rao leads the Helix Lab.", ["Aanya Rao"]),
 ("What is the lab's budget?", "INSUFFICIENT EVIDENCE", "The corpus does not state a budget; abstaining is correct.", []),
]
def token_correct(candidate, expected):
    a = candidate.lower()
    return bool(expected) and all(t.lower() in a for t in expected)
print("fixture                                   token   judge.correct  judge.faithful  reason")
for q,cand,ref,exp in fixtures:
    j=llm_judge(q,cand,ref)
    tok=token_correct(cand,exp)
    print(f"  {cand[:38]:40} {str(tok):6}  {str(j['correct']):12}  {str(j['faithful']):13}  {j['reason'][:40]}")
print("\nExpect: paraphrase -> token FALSE but judge correct TRUE; verbose-fabrication ->")
print("token TRUE but judge faithful FALSE; abstention -> token FALSE but judge correct TRUE.")
print("Token-presence and the judge disagree exactly where token-presence is weak.")

## Step 3: The drop-in scorer for Lab 34

Same signature as Lab 34's `answer_correct`; swap it into `run_harness` to upgrade the head-to-head.

In [ ]:
# The drop-in for Lab 34. Lab 34 scored with `answer_correct` (expected-token presence).
# Here is the judge-backed scorer with the same signature - swap it into Lab 34's
# run_harness to upgrade the whole head-to-head from token-presence to LLM-judged.
def answer_correct_judge(answer, item):
    reference = ("This question is unanswerable from the corpus; abstaining is correct."
                 if item["expected_behavior"]=="abstain"
                 else "Expected facts: " + ", ".join(item["expected_contains"]))
    return llm_judge(item["query"], answer, reference)["correct"]

# Demonstrate on Lab 34's eval set with a single condensed pipeline (static), comparing
# both scorers. Scoring all four patterns = plug answer_correct_judge into Lab 34.
with open("../34-rag-pattern-head-to-head/eval_set.jsonl") as f:
    eval_set = [json.loads(line) for line in f]
print(f"Loaded {len(eval_set)} eval queries. Swap answer_correct_judge into Lab 34's")
print("run_harness to re-score static/CRAG/Self-RAG/Graph with the judge instead of tokens.")

## Step 4: Judge caveats (read before trusting scores)

The judge is a measurement instrument with bias and noise.

In [ ]:
# Before you trust judge scores as a metric, know the failure modes:
#  - position & verbosity bias: judges favor longer answers and the first option in
#    pairwise mode. We use pointwise scoring with an explicit rubric to reduce this.
#  - self-preference: a judge tends to favor text from its own model family - hence a
#    different/stronger judge model than the generator.
#  - nondeterminism: even at temperature 0 there is run-to-run variance, so judged
#    metrics need a tolerance band, not exact equality.
#  - the judge has its own error rate: validate it against a small human-labeled set
#    and report judge-vs-human agreement before reporting judge-vs-pattern scores.
print("An LLM judge is a measurement instrument with bias and noise - calibrate it,")
print("don't assume it. Validate against human labels; treat scores as estimates.")

## Step 5: The eval gate (threshold logic)

Pure, testable: which metrics fall below their required minimums?

In [ ]:
# --- The eval gate: catch routing regressions in CI ---
# Pure threshold logic (testable, deterministic). The router's ROUTING accuracy is the
# BLOCKING check - it needs only the trained classifier (no LLM, no flakiness). Answer
# faithfulness via the judge runs as an informational nightly job, not a blocking gate.
def gate(metrics:
    dict, thresholds: dict):
    failures=[]
    for k,minv in thresholds.items():
        v=metrics.get(k)
        if v is None:
            failures.append(f"{k}: missing")
            continue
        if v < minv:
            failures.append(f"{k}={v:.3f} < required {minv:.3f}")
    return (len(failures)==0, failures)

# Example: a router that regressed on routing accuracy fails the gate.
thresholds={"routing_accuracy":0.85,"answer_accuracy":0.75}
good={"routing_accuracy":0.94,"answer_accuracy":0.81}
bad ={"routing_accuracy":0.69,"answer_accuracy":0.81}
for name,m in [("good",good),("bad",bad)]:
    ok,fails=gate(m,thresholds)
    print(f"{name}: {'PASS' if ok else 'FAIL'}  {fails}")

## Step 6: The runnable gate

The `eval_gate.py` CI entrypoint shipped with this lab.

In [ ]:
# A runnable gate. In CI this is `python eval_gate.py` (shipped alongside this lab):
# it loads the eval set, runs the trained router for ROUTING accuracy (deterministic),
# optionally scores ANSWER accuracy (token-presence by default, --judge for the LLM
# judge), applies thresholds, prints a report, and exits non-zero on regression.
print(pathlib.Path("./eval_gate.py").read_text()[:1500])
print("...\n[full script in this lab's directory; the workflow runs it on PRs]")

## Step 7: What blocks CI vs what gets monitored

In [ ]:
# Why routing accuracy is the blocking gate and judged faithfulness is not:
#  - routing accuracy is deterministic and cheap (classifier only) -> a stable CI gate.
#  - LLM-judged faithfulness is noisy and costs calls -> running it as a *blocking* gate
#    makes the build flaky and expensive. Run it nightly/informational, alert on trend.
# The general rule: block on cheap deterministic signals, monitor on expensive noisy ones.
print("Block CI on deterministic routing accuracy; monitor judged faithfulness nightly.")

## What you built

An LLM judge that scores correctness and faithfulness (drop-in for Lab 34's token-presence scorer) and an eval gate that blocks CI on routing-accuracy regressions. The design split is the lesson: **block on cheap deterministic signals (routing accuracy), monitor expensive noisy ones (judged faithfulness) out of band.**

**Where this simplifies:** the judge is pointwise with one rubric (pairwise judging and rubric ensembles reduce variance further); we do not ship the human-label validation set the judge should be calibrated against (build one before trusting judged metrics); the gate's thresholds are illustrative — set yours from a baseline run plus a tolerance band, not from intuition.

Files in this lab: [`eval_gate.py`](./eval_gate.py) (CI entrypoint) and the workflow at [`.github/workflows/rag-eval-gate.yml`](../../.github/workflows/rag-eval-gate.yml).